In [1]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive")
matches = list(root.rglob(
    "Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main.zip"
))

print("Arsip ditemukan:", len(matches))
for path in matches:
    print(path)

assert len(matches) == 1, "Pastikan ada tepat satu arsip benchmark."

out = Path("/content/evtol_reference")
out.mkdir(exist_ok=True)

with zipfile.ZipFile(matches[0]) as z:
    z.extractall(out)

print("Ekstraksi selesai.")

Mounted at /content/drive
Arsip ditemukan: 1
/content/drive/MyDrive/Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML-main.zip
Ekstraksi selesai.


In [2]:
import pandas as pd

files = list(out.rglob("health_mission_features_targets.csv"))
assert len(files) == 1

df = pd.read_csv(files[0])

print("Jumlah sel:", df["cell_id"].nunique())
print("Daftar sel:")
print(sorted(df["cell_id"].dropna().unique()))

print("\nJumlah baris per sel:")
print(df.groupby("cell_id").size().to_string())

print("\nKolom target:")
print([
    c for c in df.columns
    if "SOH" in c or "RUL" in c
])

Jumlah sel: 19
Daftar sel:
['VAH01', 'VAH02', 'VAH05', 'VAH10', 'VAH11', 'VAH12', 'VAH13', 'VAH15', 'VAH16', 'VAH17', 'VAH20', 'VAH22', 'VAH23', 'VAH24', 'VAH25', 'VAH26', 'VAH27', 'VAH28', 'VAH30']

Jumlah baris per sel:
cell_id
VAH01     829
VAH02     611
VAH05    1516
VAH10    1400
VAH11    2203
VAH12    2302
VAH13    1020
VAH15     542
VAH16     547
VAH17     981
VAH20     598
VAH22     540
VAH23     682
VAH24     784
VAH25     541
VAH26    1138
VAH27     573
VAH28    1155
VAH30     900

Kolom target:
['SOH_end_pct', 'RUL90_event_mission', 'RUL90_observed', 'RUL90_missions', 'RUL85_event_mission', 'RUL85_observed', 'RUL85_missions', 'RUL80_event_mission', 'RUL80_observed', 'RUL80_missions', 'SOH_m_plus_5_pct', 'SOH_drop_5_pct']


In [3]:
pred_files = list(out.rglob("sequence_predictions.csv"))
assert len(pred_files) == 1

pred = pd.read_csv(pred_files[0])

summary = (
    pred.groupby(["task", "model"])
    .agg(
        observations=("y_true", "size"),
        cells=("test_cell", "nunique")
    )
    .reset_index()
)

display(summary)

print("\nMissing values:")
print(pred[
    ["task", "model", "test_cell", "y_true", "y_pred"]
].isna().sum())

,task,model,observations,cells
0,soh,mission_phase_transformer,18513,19



Missing values:
task         0
model        0
test_cell    0
y_true       0
y_pred       0
dtype: int64


In [4]:
audit_dir = root / "evtol_revision_audit"
audit_dir.mkdir(exist_ok=True)

summary.to_csv(
    audit_dir / "prediction_inventory.csv",
    index=False
)

df.groupby("cell_id").size().rename(
    "n_rows"
).to_csv(audit_dir / "cell_inventory.csv")

print("Audit tersimpan di:", audit_dir)

Audit tersimpan di: /content/drive/MyDrive/evtol_revision_audit


In [5]:
import pandas as pd

fold_files = list(out.rglob("sequence_fold_metrics.csv"))
assert len(fold_files) == 1, "File fold metrics tidak ditemukan"

folds = pd.read_csv(fold_files[0])

print("Kolom tersedia:")
print(folds.columns.tolist())

display(folds.head())

Kolom tersedia:
['task', 'model', 'fold', 'test_cell', 'train_cells', 'validation_cells', 'train_n', 'validation_n', 'test_n', 'test_mae', 'test_rmse', 'inference_ms_per_sample', 'best_epoch', 'best_val_mae', 'epochs_run', 'training_seconds', 'parameter_count', 'target_mean', 'target_std']


,task,model,fold,test_cell,train_cells,validation_cells,train_n,validation_n,test_n,test_mae,test_rmse,inference_ms_per_sample,best_epoch,best_val_mae,epochs_run,training_seconds,parameter_count,target_mean,target_std
0,soh,mission_phase_transformer,1,VAH01,VAH05;VAH10;VAH11;VAH13;VAH15;VAH16;VAH17;VAH2...,VAH02;VAH12;VAH25;VAH26;VAH27;VAH30,11744,5945,824,4.306988,5.243617,0.036401,15,1.182358,23,112.886770,160417,86.692734,6.234044
1,soh,mission_phase_transformer,2,VAH02,VAH01;VAH05;VAH10;VAH11;VAH13;VAH15;VAH17;VAH2...,VAH12;VAH16;VAH23;VAH26;VAH27;VAH30,11842,6065,606,0.488714,0.626200,0.036162,5,1.238756,13,64.243190,160417,86.583946,6.188626
2,soh,mission_phase_transformer,3,VAH05,VAH01;VAH10;VAH11;VAH15;VAH16;VAH17;VAH20;VAH2...,VAH02;VAH12;VAH13;VAH25;VAH27;VAH30,11211,5827,1475,1.137688,1.250389,0.054845,3,0.736543,11,51.385701,160417,86.893272,6.203854
3,soh,mission_phase_transformer,4,VAH10,VAH01;VAH05;VAH11;VAH15;VAH16;VAH17;VAH20;VAH2...,VAH02;VAH12;VAH13;VAH25;VAH27;VAH30,11291,5827,1395,1.351763,1.525677,0.057797,10,0.835513,18,86.533066,160417,87.102577,6.049048
4,soh,mission_phase_transformer,5,VAH11,VAH05;VAH10;VAH15;VAH16;VAH17;VAH20;VAH23;VAH2...,VAH01;VAH02;VAH12;VAH13;VAH22;VAH25,10631,5684,2198,2.043026,2.296876,0.050849,25,1.531009,33,149.043245,160417,87.451149,5.645945


In [6]:
required = [
    "task", "model", "test_cell",
    "mission_id", "y_true", "y_pred"
]

assert all(c in pred.columns for c in required)

duplicates = pred.duplicated(
    subset=["task", "model", "test_cell", "mission_id"]
)

print("Jumlah prediksi duplikat:", duplicates.sum())

print("\nJumlah sel per tugas dan model:")
display(
    pred.groupby(["task", "model"])["test_cell"]
    .nunique()
    .reset_index(name="n_cells")
)

print("\nJumlah observasi per tugas dan model:")
display(
    pred.groupby(["task", "model"])
    .size()
    .reset_index(name="n_observations")
)

Jumlah prediksi duplikat: 0

Jumlah sel per tugas dan model:


,task,model,n_cells
0,soh,mission_phase_transformer,19



Jumlah observasi per tugas dan model:


,task,model,n_observations
0,soh,mission_phase_transformer,18513


In [7]:
audit_dir.mkdir(parents=True, exist_ok=True)

folds.to_csv(
    audit_dir / "fold_metadata_audit.csv",
    index=False
)

pred.groupby(["task", "model", "test_cell"]).size().rename(
    "n_predictions"
).to_csv(
    audit_dir / "predictions_per_cell.csv"
)

print("File audit berhasil disimpan.")

File audit berhasil disimpan.


In [8]:
def parse_cells(value):
    if pd.isna(value):
        return set()
    return {
        x.strip()
        for x in str(value).split(";")
        if x.strip()
    }

audit_rows = []

for _, row in folds.iterrows():
    train = parse_cells(row["train_cells"])
    val = parse_cells(row["validation_cells"])
    test = {str(row["test_cell"]).strip()}

    audit_rows.append({
        "task": row["task"],
        "model": row["model"],
        "fold": row["fold"],
        "test_cell": row["test_cell"],
        "train_val_overlap": len(train & val),
        "train_test_overlap": len(train & test),
        "val_test_overlap": len(val & test),
        "n_train_cells": len(train),
        "n_val_cells": len(val)
    })

leakage_audit = pd.DataFrame(audit_rows)

display(leakage_audit.head())

print("\nJumlah fold dengan overlap:")
print(
    (leakage_audit[
        ["train_val_overlap",
         "train_test_overlap",
         "val_test_overlap"]
    ] > 0).sum()
)

print("\nJumlah fold:", len(leakage_audit))
print("Model:", leakage_audit["model"].unique())
print("Task:", leakage_audit["task"].unique())

,task,model,fold,test_cell,train_val_overlap,train_test_overlap,val_test_overlap,n_train_cells,n_val_cells
0,soh,mission_phase_transformer,1,VAH01,0,0,0,12,6
1,soh,mission_phase_transformer,2,VAH02,0,0,0,12,6
2,soh,mission_phase_transformer,3,VAH05,0,0,0,12,6
3,soh,mission_phase_transformer,4,VAH10,0,0,0,12,6
4,soh,mission_phase_transformer,5,VAH11,0,0,0,12,6



Jumlah fold dengan overlap:
train_val_overlap     0
train_test_overlap    0
val_test_overlap      0
dtype: int64

Jumlah fold: 19
Model: ['mission_phase_transformer']
Task: ['soh']


In [9]:
import numpy as np

errors = pred["y_true"] - pred["y_pred"]

mae = np.abs(errors).mean()
rmse = np.sqrt(np.mean(errors ** 2))

cell_mae = (
    pred.assign(abs_error=np.abs(errors))
    .groupby("test_cell")["abs_error"]
    .mean()
)

print("Pooled MAE:", round(mae, 6))
print("Pooled RMSE:", round(rmse, 6))
print("Macro-MAE:", round(cell_mae.mean(), 6))
print("Worst-cell MAE:", round(cell_mae.max(), 6))

Pooled MAE: 1.251896
Pooled RMSE: 2.111085
Macro-MAE: 1.244602
Worst-cell MAE: 4.306988


In [10]:
import numpy as np
import pandas as pd

# Hitung metrik per sel dari file prediksi
cell_metrics = (
    pred.assign(
        abs_error=(pred["y_true"] - pred["y_pred"]).abs(),
        squared_error=(pred["y_true"] - pred["y_pred"]) ** 2
    )
    .groupby(["task", "model", "test_cell"])
    .agg(
        pred_n=("y_true", "size"),
        pred_mae=("abs_error", "mean"),
        mean_squared_error=("squared_error", "mean")
    )
    .reset_index()
)

cell_metrics["pred_rmse"] = np.sqrt(
    cell_metrics["mean_squared_error"]
)

# Cocokkan dengan laporan evaluasi per fold
comparison = folds.merge(
    cell_metrics,
    on=["task", "model", "test_cell"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

comparison["mae_difference"] = (
    comparison["pred_mae"] - comparison["test_mae"]
).abs()

comparison["rmse_difference"] = (
    comparison["pred_rmse"] - comparison["test_rmse"]
).abs()

print("Status pencocokan:")
print(comparison["_merge"].value_counts())

print("\nSelisih MAE terbesar:")
print(comparison["mae_difference"].max())

print("\nSelisih RMSE terbesar:")
print(comparison["rmse_difference"].max())

print("\nFold dengan selisih terbesar:")
display(
    comparison.sort_values(
        "mae_difference", ascending=False
    )[
        [
            "fold", "test_cell",
            "test_mae", "pred_mae",
            "test_rmse", "pred_rmse",
            "mae_difference", "rmse_difference"
        ]
    ].head(10)
)

comparison.to_csv(
    audit_dir / "fold_prediction_consistency.csv",
    index=False
)

Status pencocokan:
_merge
both          19
left_only      0
right_only     0
Name: count, dtype: int64

Selisih MAE terbesar:
3.155445580382832e-07

Selisih RMSE terbesar:
3.7519859130696886e-07

Fold dengan selisih terbesar:


,fold,test_cell,test_mae,pred_mae,test_rmse,pred_rmse,mae_difference,rmse_difference
16,17,VAH27,1.644099,1.644098,6.493909,6.493909,3.155446e-07,3.751986e-07
13,14,VAH24,1.351140,1.351140,1.667547,1.667547,1.967108e-07,5.750626e-08
11,12,VAH22,0.576553,0.576554,0.670757,0.670757,1.875315e-07,8.926834e-08
8,9,VAH16,1.382682,1.382683,1.418154,1.418154,1.856177e-07,1.275831e-07
5,6,VAH12,0.718999,0.718999,0.859042,0.859043,1.568948e-07,1.010567e-07
14,15,VAH25,0.382784,0.382784,0.438385,0.438385,1.203146e-07,1.090861e-08
6,7,VAH13,0.540153,0.540153,0.835441,0.835441,1.172187e-07,1.606857e-08
17,18,VAH28,0.228713,0.228713,0.273545,0.273545,1.157992e-07,1.132136e-07
10,11,VAH20,2.030605,2.030605,2.221573,2.221573,1.054231e-07,4.663847e-08
3,4,VAH10,1.351763,1.351763,1.525677,1.525677,9.347024e-08,8.022103e-08


In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

records = []

for path in out.rglob("*.csv"):
    try:
        header = pd.read_csv(path, nrows=0)
        columns = set(header.columns)

        if {"y_true", "y_pred"}.issubset(columns):
            df_check = pd.read_csv(path)

            if {"task", "model"}.issubset(df_check.columns):
                groups = df_check.groupby(["task", "model"])
            else:
                groups = [(("unknown", "unknown"), df_check)]

            for (task, model), group in groups:
                valid = group[["y_true", "y_pred"]].dropna()
                if valid.empty:
                    continue

                err = valid["y_true"] - valid["y_pred"]

                records.append({
                    "file": str(path.relative_to(out)),
                    "task": task,
                    "model": model,
                    "n": len(valid),
                    "mae": np.abs(err).mean(),
                    "rmse": np.sqrt(np.mean(err**2))
                })
    except (ValueError, KeyError, UnicodeError, pd.errors.ParserError) as exc:
        print("File dilewati:", path.name, type(exc).__name__)

inventory = pd.DataFrame(records)

if inventory.empty:
    print("Tidak ditemukan file CSV lain dengan kolom y_true dan y_pred.")
else:
    display(
        inventory.sort_values(["task", "model", "file"])
    )
    inventory.to_csv(
        audit_dir / "prediction_source_inventory.csv",
        index=False
    )

,file,task,model,n,mae,rmse
0,Mission-Phase-Feature-Learning-for-eVTOL-Batte...,soh,mission_phase_transformer,18513,1.251896,2.111085


In [12]:
print("Jumlah file CSV dalam arsip:",
      len(list(out.rglob("*.csv"))))

print("\nModel dan tugas dalam prediksi:")
display(
    pred.groupby(["task", "model"])
    .size()
    .reset_index(name="n_predictions")
)

Jumlah file CSV dalam arsip: 8

Model dan tugas dalam prediksi:


,task,model,n_predictions
0,soh,mission_phase_transformer,18513


In [13]:
from pathlib import Path
import pandas as pd

drive_root = Path("/content/drive/MyDrive")

keywords = [
    "prediction", "predictions",
    "soh", "rul",
    "fold", "metrics",
    "result", "summary",
    "lightgbm", "xgboost",
    "catboost", "random_forest",
    "transformer", "moe"
]

matches = []

for path in drive_root.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {
        ".csv", ".parquet", ".json", ".xlsx"
    }:
        continue

    if any(k in path.name.lower() for k in keywords):
        matches.append({
            "filename": path.name,
            "path": str(path),
            "size_mb": round(path.stat().st_size / 1024**2, 3)
        })

found = pd.DataFrame(matches)

print("Jumlah file kandidat:", len(found))

if not found.empty:
    display(
        found.sort_values("filename").head(100)
    )

Jumlah file kandidat: 427


,filename,path,size_mb
2,actrajnet_7days1_epoch1_altitude_summary.csv,/content/drive/MyDrive/ACTrajNet_backup/actraj...,0.000
1,actrajnet_7days1_epoch1_predictions.csv,/content/drive/MyDrive/ACTrajNet_backup/actraj...,52.020
113,catboost_feature_manifest.json,/content/drive/MyDrive/eVTOL_table5_work/final...,0.001
103,catboost_fold_metrics.csv,/content/drive/MyDrive/eVTOL_table5_work/final...,0.001
114,catboost_loco_fold_metrics.csv,/content/drive/MyDrive/eVTOL_table5_work/final...,0.001
...,...,...,...
421,fold_record.json,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,0.001
419,fold_record.json,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,0.001
417,fold_record.json,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,0.001
415,fold_record.json,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,0.001


In [14]:
# Fokus pada eksperimen eVTOL
evtol = found[
    found["path"].str.contains("evtol", case=False, na=False)
].copy()

# Prioritaskan file prediksi, ringkasan hasil, dan konfigurasi
pattern = (
    r"prediction|predictions|"
    r"oof|held.?out|"
    r"summary|metrics|"
    r"manifest|config|"
    r"fold_record"
)

candidates = evtol[
    evtol["filename"].str.contains(
        pattern, case=False, regex=True, na=False
    )
].copy()

# Tampilkan direktori, bukan 427 baris sekaligus
candidates["folder"] = candidates["path"].apply(
    lambda p: str(Path(p).parent)
)

folder_summary = (
    candidates.groupby("folder")
    .agg(
        n_files=("filename", "size"),
        total_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values("n_files", ascending=False)
)

print("Jumlah folder kandidat:", len(folder_summary))
display(folder_summary.head(30))

Jumlah folder kandidat: 168


,folder,n_files,total_mb
161,/content/drive/MyDrive/eVTOL_table5_work/final...,12,4.053
160,/content/drive/MyDrive/eVTOL_table5_work/final...,10,3.786
139,/content/drive/MyDrive/eVTOL_row_random_recove...,7,3.116
5,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,5,6.251
167,/content/drive/MyDrive/evtol_revision_audit,4,0.009
0,/content/drive/MyDrive/eVTOL_Table3_recovery_b...,3,1.226
3,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,3,0.050
1,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,3,1.226
130,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,3,0.012
140,/content/drive/MyDrive/eVTOL_table4_work/seque...,3,1.102


In [15]:
prediction_inventory = []

for file_path in candidates["path"]:
    path = Path(file_path)

    if path.suffix.lower() != ".csv":
        continue

    try:
        header = pd.read_csv(path, nrows=0)
        columns = header.columns.tolist()
        lower = [c.lower() for c in columns]

        has_true = any(
            k in lower for k in
            ["y_true", "true", "actual", "target", "soh_true"]
        )
        has_pred = any(
            k in lower for k in
            ["y_pred", "prediction", "predicted", "soh_pred"]
        )

        if has_true and has_pred:
            prediction_inventory.append({
                "file": path.name,
                "path": str(path),
                "columns": ", ".join(columns[:15]),
                "size_mb": round(
                    path.stat().st_size / 1024**2, 3
                )
            })

    except (OSError, UnicodeError, pd.errors.ParserError):
        continue

prediction_files = pd.DataFrame(prediction_inventory)

print("File prediksi potensial:", len(prediction_files))

if not prediction_files.empty:
    display(prediction_files)

File prediksi potensial: 174


,file,path,columns,size_mb
0,sequence_predictions.csv,/content/drive/MyDrive/eVTOL_Table3_recovery_b...,"task, model, test_cell, mission_id, y_true, y_...",1.219
1,soh_rf_fold1_predictions.csv,/content/drive/MyDrive/eVTOL_row_random_recove...,"task, model, protocol, fold, cell, mission_id,...",0.315
2,soh_rf_fold2_predictions.csv,/content/drive/MyDrive/eVTOL_row_random_recove...,"task, model, protocol, fold, cell, mission_id,...",0.315
3,soh_rf_fold3_predictions.csv,/content/drive/MyDrive/eVTOL_row_random_recove...,"task, model, protocol, fold, cell, mission_id,...",0.314
4,soh_rf_fold4_predictions.csv,/content/drive/MyDrive/eVTOL_row_random_recove...,"task, model, protocol, fold, cell, mission_id,...",0.314
...,...,...,...,...
169,predictions.csv,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,"task, model, test_cell, mission_id, y_true, y_...",0.034
170,predictions.csv,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,"task, model, test_cell, mission_id, y_true, y_...",0.069
171,predictions.csv,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,"task, model, test_cell, mission_id, y_true, y_...",0.052
172,predictions.csv,/content/drive/MyDrive/eVTOL_benchmark_reprodu...,"cell_id, mission_id, y_true, y_pred, error",0.011


In [16]:
from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive")

target_folders = [
    "eVTOL_manuscript_results",
    "eVTOL_table5_work",
    "eVTOL_benchmark_reproduction_v2",
    "eVTOL_table4_work",
]

rows = []

for folder_name in target_folders:
    folder = root / folder_name

    print(f"\n{'=' * 60}")
    print(folder_name)
    print("ADA" if folder.exists() else "TIDAK DITEMUKAN")

    if not folder.exists():
        continue

    for p in folder.rglob("*"):
        if not p.is_file():
            continue

        if p.suffix.lower() not in {
            ".csv", ".json", ".parquet", ".xlsx"
        }:
            continue

        rows.append({
            "source_folder": folder_name,
            "relative_path": str(p.relative_to(folder)),
            "size_mb": round(p.stat().st_size / 1024**2, 3)
        })

files_target = pd.DataFrame(rows)

if not files_target.empty:
    # Tampilkan file di luar folder fold individual terlebih dahulu
    priority = files_target[
        files_target["relative_path"].str.contains(
            r"summary|combined|pooled|manifest|"
            r"aggregate|all_models|final|metrics",
            case=False,
            regex=True,
            na=False
        )
    ]

    print("\nFILE PRIORITAS:")
    display(priority.head(80))

    print("\nJUMLAH FILE PER FOLDER:")
    display(
        files_target.groupby("source_folder")
        .size()
        .reset_index(name="n_files")
    )


eVTOL_manuscript_results
ADA

eVTOL_table5_work
ADA

eVTOL_benchmark_reproduction_v2
ADA

eVTOL_table4_work
ADA

FILE PRIORITAS:


,source_folder,relative_path,size_mb
0,eVTOL_manuscript_results,artifact_manifest_sha256.csv,0.002
2,eVTOL_manuscript_results,soh_six_model/table1_six_model_soh_metrics.csv,0.001
8,eVTOL_manuscript_results,rul_two_model/table_rul_model_metrics.csv,0.001
34,eVTOL_manuscript_results,manuscript/tree_manifest_metadata_verification...,0.003
35,eVTOL_manuscript_results,manuscript/results_3_1_metrics_snapshot.csv,0.001
71,eVTOL_table5_work,tree_soh/table5_soh_partial_metrics.csv,0.000
72,eVTOL_table5_work,tree_soh/table5_soh_partial_per_cell_metrics.csv,0.002
73,eVTOL_table5_work,final_fast/random_forest_fold_metrics.csv,0.001
74,eVTOL_table5_work,final_fast/random_forest_predictions.csv,1.055
75,eVTOL_table5_work,final_fast/xgboost_fold_metrics.csv,0.001



JUMLAH FILE PER FOLDER:


,source_folder,n_files
0,eVTOL_benchmark_reproduction_v2,526
1,eVTOL_manuscript_results,39
2,eVTOL_table4_work,81
3,eVTOL_table5_work,58


In [17]:
from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive")
base = root / "eVTOL_manuscript_results"

targets = {
    "SOH enam model":
        base / "soh_six_model/table1_six_model_soh_metrics.csv",

    "RUL dua model":
        base / "rul_two_model/table_rul_model_metrics.csv",

    "Snapshot manuskrip":
        base / "manuscript/results_3_1_metrics_snapshot.csv",

    "Manifest artefak":
        base / "artifact_manifest_sha256.csv"
}

for label, path in targets.items():
    print("\n" + "=" * 65)
    print(label)
    print("Path:", path)

    if not path.exists():
        print("STATUS: FILE TIDAK DITEMUKAN")
        continue

    df_check = pd.read_csv(path)

    print("Ukuran:", df_check.shape)
    print("Kolom:", df_check.columns.tolist())
    display(df_check.head(30))


SOH enam model
Path: /content/drive/MyDrive/eVTOL_manuscript_results/soh_six_model/table1_six_model_soh_metrics.csv
Ukuran: (6, 11)
Kolom: ['model', 'samples', 'cells', 'pooled_MAE', 'pooled_RMSE', 'macro_MAE', 'median_cell_MAE', 'P90_cell_MAE', 'worst_cell_MAE', 'mean_signed_error', 'overprediction_pct']


,model,samples,cells,pooled_MAE,pooled_RMSE,macro_MAE,median_cell_MAE,P90_cell_MAE,worst_cell_MAE,mean_signed_error,overprediction_pct
0,LightGBM,18513,19,0.525101,0.759373,0.544597,0.480593,0.904026,1.924715,0.056997,55.447523
1,XGBoost,18513,19,0.571072,0.824290,0.582537,0.522556,0.927922,2.094553,0.083704,57.203047
2,Random Forest,18513,19,0.599901,0.852420,0.591775,0.477539,0.955130,1.823650,0.041440,54.156539
3,CatBoost,18513,19,0.804183,1.324482,0.712724,0.522814,1.190496,2.952953,0.225358,52.287582
4,Mission-Phase Transformer,18513,19,1.374355,2.253690,1.375024,0.844629,3.108116,5.456126,0.064425,56.878950
5,Attention-LSTM-MoE,18513,19,1.434482,2.147080,1.488137,0.872558,2.932263,5.427888,0.223312,58.132123



RUL dua model
Path: /content/drive/MyDrive/eVTOL_manuscript_results/rul_two_model/table_rul_model_metrics.csv
Ukuran: (6, 11)
Kolom: ['task', 'model', 'samples', 'cells', 'pooled_MAE', 'pooled_RMSE', 'macro_MAE', 'median_cell_MAE', 'P90_cell_MAE', 'worst_cell_MAE', 'mean_signed_error']


,task,model,samples,cells,pooled_MAE,pooled_RMSE,macro_MAE,median_cell_MAE,P90_cell_MAE,worst_cell_MAE,mean_signed_error
0,RUL90,Mission-Phase Transformer,5814,19,36.869410,67.286792,34.818092,17.449477,68.283558,186.662622,-15.753896
1,RUL90,Attention-LSTM-MoE,5814,19,37.169595,55.168945,35.358778,23.535114,70.598267,106.042674,-2.035238
2,RUL85,Mission-Phase Transformer,9595,16,73.475964,95.474907,73.104311,58.819925,153.661855,211.215465,18.456540
3,RUL85,Attention-LSTM-MoE,9595,16,67.478452,89.400861,66.585784,55.345312,137.219900,197.008325,26.754870
4,RUL80,Mission-Phase Transformer,8257,8,175.467016,233.326444,160.021477,143.636839,339.472331,410.129417,-0.196581
5,RUL80,Attention-LSTM-MoE,8257,8,180.636283,222.322977,167.873754,157.311117,304.708373,387.016327,31.364912



Snapshot manuskrip
Path: /content/drive/MyDrive/eVTOL_manuscript_results/manuscript/results_3_1_metrics_snapshot.csv
Ukuran: (6, 11)
Kolom: ['model', 'samples', 'cells', 'pooled_MAE', 'pooled_RMSE', 'macro_MAE', 'median_cell_MAE', 'P90_cell_MAE', 'worst_cell_MAE', 'mean_signed_error', 'overprediction_pct']


,model,samples,cells,pooled_MAE,pooled_RMSE,macro_MAE,median_cell_MAE,P90_cell_MAE,worst_cell_MAE,mean_signed_error,overprediction_pct
0,LightGBM,18513,19,0.525101,0.759373,0.544597,0.480593,0.904026,1.924715,0.056997,55.447523
1,XGBoost,18513,19,0.571072,0.824290,0.582537,0.522556,0.927922,2.094553,0.083704,57.203047
2,Random Forest,18513,19,0.599901,0.852420,0.591775,0.477539,0.955130,1.823650,0.041440,54.156539
3,CatBoost,18513,19,0.804183,1.324482,0.712724,0.522814,1.190496,2.952953,0.225358,52.287582
4,Mission-Phase Transformer,18513,19,1.374355,2.253690,1.375024,0.844629,3.108116,5.456126,0.064425,56.878950
5,Attention-LSTM-MoE,18513,19,1.434482,2.147080,1.488137,0.872558,2.932263,5.427888,0.223312,58.132123



Manifest artefak
Path: /content/drive/MyDrive/eVTOL_manuscript_results/artifact_manifest_sha256.csv
Ukuran: (12, 5)
Kolom: ['category', 'filename', 'relative_path', 'size_bytes', 'sha256']


,category,filename,relative_path,size_bytes,sha256
0,SOH,table1_six_model_soh_metrics.csv,soh_six_model/table1_six_model_soh_metrics.csv,1181,421d0b414275841f80dda7b635562f9374979699710f4d...
1,SOH,table2_wilcoxon_vs_lightgbm.csv,soh_six_model/table2_wilcoxon_vs_lightgbm.csv,792,52bb586eef966fe8dba22c0ce51dbcac456b35c489069f...
2,SOH,table3_low_vs_high_soh.csv,soh_six_model/table3_low_vs_high_soh.csv,841,4f9dca4c72f693b11f8aa91465766febfa3fc327c3b571...
3,SOH,supplement_cell_level_mae.csv,soh_six_model/supplement_cell_level_mae.csv,2382,e5497ef3cab8da3cf0f4bed44211e00f4796ab64f893cd...
4,SOH,supplement_cell_soh_strata.csv,soh_six_model/supplement_cell_soh_strata.csv,33324,ac0f1b91668db7ff2dd363bc78499819177cf29b63d91d...
5,SOH,supplement_macro_mae_by_soh.csv,soh_six_model/supplement_macro_mae_by_soh.csv,599,3cdf0eaa5d954e927c7a2004985eec4445918f9d1691a9...
6,SOH,figure1_cell_mae_distribution.png,soh_six_model/figure1_cell_mae_distribution.png,198206,92c1e24c8304c502079934f0c9627b0c130da4f9655eb0...
7,SOH,figure2_macro_mae_by_soh.png,soh_six_model/figure2_macro_mae_by_soh.png,151376,c36e1c4a12fb92b89c28f6c0a6d2f6ec82b79fd848c33f...
8,SOH,figure3_lightgbm_low_vs_high_soh.png,soh_six_model/figure3_lightgbm_low_vs_high_soh...,176068,698baf6245598987a8355bc38e43692651e6fd0948d704...
9,RUL,table_rul_model_metrics.csv,rul_two_model/table_rul_model_metrics.csv,1114,9776fbf75e989150a81a25dbe8ceb4c7606dcc4c190e37...


In [18]:
from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive")

folders = [
    root / "eVTOL_manuscript_results/soh_six_model",
    root / "eVTOL_manuscript_results/rul_two_model",
    root / "eVTOL_benchmark_reproduction_v2/table3_full_run",
]

for folder in folders:
    print("\n" + "=" * 65)
    print("FOLDER:", folder)
    print("ADA:", folder.exists())

    if not folder.exists():
        continue

    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower()
        in {".csv", ".json", ".parquet"}
    )

    print("JUMLAH FILE:", len(files))

    for p in files:
        print(
            str(p.relative_to(folder)),
            "|",
            round(p.stat().st_size / 1024**2, 3),
            "MB"
        )


FOLDER: /content/drive/MyDrive/eVTOL_manuscript_results/soh_six_model
ADA: True
JUMLAH FILE: 6
supplement_cell_level_mae.csv | 0.002 MB
supplement_cell_soh_strata.csv | 0.032 MB
supplement_macro_mae_by_soh.csv | 0.001 MB
table1_six_model_soh_metrics.csv | 0.001 MB
table2_wilcoxon_vs_lightgbm.csv | 0.001 MB
table3_low_vs_high_soh.csv | 0.001 MB

FOLDER: /content/drive/MyDrive/eVTOL_manuscript_results/rul_two_model
ADA: True
JUMLAH FILE: 3
supplement_rul_cell_mae.csv | 0.005 MB
table_rul_model_metrics.csv | 0.001 MB
table_rul_paired_statistics.csv | 0.001 MB

FOLDER: /content/drive/MyDrive/eVTOL_benchmark_reproduction_v2/table3_full_run
ADA: True
JUMLAH FILE: 503
combined_prediction_metrics.csv | 0.001 MB
rul80/attention_lstm_moe/VAH05/fold_record.json | 0.001 MB
rul80/attention_lstm_moe/VAH05/predictions.csv | 0.071 MB
rul80/attention_lstm_moe/VAH05/scaler.json | 0.001 MB
rul80/attention_lstm_moe/VAH05/training_log.csv | 0.001 MB
rul80/attention_lstm_moe/VAH10/fold_record.json | 0.001 

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path(
    "/content/drive/MyDrive/"
    "eVTOL_benchmark_reproduction_v2/table3_full_run"
)

path = base / "sequence_predictions.csv"

assert path.exists(), f"File tidak ditemukan: {path}"

seq = pd.read_csv(path)

print("Kolom:", seq.columns.tolist())
print("Total baris:", len(seq))
display(seq.head(3))

required = {
    "task", "model", "test_cell",
    "mission_id", "y_true", "y_pred"
}
assert required.issubset(seq.columns), (
    f"Kolom kurang: {required - set(seq.columns)}"
)

assert seq[["y_true", "y_pred"]].notna().all().all()

seq["abs_error"] = (
    seq["y_true"] - seq["y_pred"]
).abs()

seq["squared_error"] = (
    seq["y_true"] - seq["y_pred"]
) ** 2

# Metrik agregat tiap tugas dan model
summary = (
    seq.groupby(["task", "model"])
    .agg(
        n=("y_true", "size"),
        cells=("test_cell", "nunique"),
        pooled_MAE=("abs_error", "mean"),
        mean_squared_error=("squared_error", "mean")
    )
    .reset_index()
)

summary["pooled_RMSE"] = np.sqrt(
    summary["mean_squared_error"]
)

# Macro-MAE dan worst-cell MAE
per_cell = (
    seq.groupby(["task", "model", "test_cell"])
    ["abs_error"]
    .mean()
    .reset_index(name="cell_MAE")
)

cell_summary = (
    per_cell.groupby(["task", "model"])
    .agg(
        macro_MAE=("cell_MAE", "mean"),
        worst_cell_MAE=("cell_MAE", "max")
    )
    .reset_index()
)

summary = summary.merge(
    cell_summary,
    on=["task", "model"],
    validate="one_to_one"
)

summary = summary.drop(
    columns="mean_squared_error"
)

print("\nHASIL HITUNG ULANG:")
display(summary.round(6))

print("\nJUMLAH DUPLIKAT:")
print(
    seq.duplicated(
        ["task", "model", "test_cell", "mission_id"]
    ).sum()
)

summary.to_csv(
    audit_dir / "table3_full_run_recomputed_metrics.csv",
    index=False
)

Kolom: ['task', 'model', 'test_cell', 'mission_id', 'y_true', 'y_pred', 'error']
Total baris: 84358


,task,model,test_cell,mission_id,y_true,y_pred,error
0,soh,attention_lstm_moe,VAH01,1,99.65223,85.79429,-13.857941
1,soh,attention_lstm_moe,VAH01,2,99.59427,85.40152,-14.192749
2,soh,attention_lstm_moe,VAH01,3,99.53631,85.47368,-14.062630



HASIL HITUNG ULANG:


,task,model,n,cells,pooled_MAE,pooled_RMSE,macro_MAE,worst_cell_MAE
0,rul80,attention_lstm_moe,8257,8,180.636283,222.322977,167.873755,387.016325
1,rul80,mission_phase_transformer,8257,8,175.467016,233.326444,160.021477,410.129418
2,rul85,attention_lstm_moe,9595,16,67.478453,89.400861,66.585784,197.008325
3,rul85,mission_phase_transformer,9595,16,73.475964,95.474907,73.104311,211.215464
4,rul90,attention_lstm_moe,5814,19,37.169595,55.168945,35.358778,106.042674
5,rul90,mission_phase_transformer,5814,19,36.869410,67.286792,34.818092,186.662622
6,soh,attention_lstm_moe,18513,19,1.434482,2.147080,1.488137,5.427888
7,soh,mission_phase_transformer,18513,19,1.374355,2.253690,1.375024,5.456126



JUMLAH DUPLIKAT:
0


In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path(
    "/content/drive/MyDrive/"
    "eVTOL_table5_work/final_fast_loco_fixed"
)

files = {
    "Random Forest": "random_forest_predictions.csv",
    "XGBoost": "xgboost_predictions.csv",
    "LightGBM": "lightgbm_predictions.csv",
    "CatBoost": "catboost_loco_predictions.csv"
}

for model, filename in files.items():
    path = base / filename

    print("\nMODEL:", model)

    if not path.exists():
        print("FILE TIDAK DITEMUKAN:", filename)
        continue

    df = pd.read_csv(path)

    print("Jumlah baris:", len(df))
    print("Kolom:", df.columns.tolist())
    display(df.head(2))


MODEL: Random Forest
Jumlah baris: 18513
Kolom: ['cell_id', 'mission_id', 'model', 'y_true', 'y_pred', 'fold']


,cell_id,mission_id,model,y_true,y_pred,fold
0,VAH01,1,Random Forest,99.652231,99.630412,1
1,VAH01,2,Random Forest,99.594270,99.573746,1



MODEL: XGBoost
Jumlah baris: 18513
Kolom: ['cell_id', 'mission_id', 'model', 'y_true', 'y_pred', 'fold']


,cell_id,mission_id,model,y_true,y_pred,fold
0,VAH01,1,XGBoost,99.652231,99.557130,1
1,VAH01,2,XGBoost,99.594270,99.473175,1



MODEL: LightGBM
Jumlah baris: 18513
Kolom: ['cell_id', 'mission_id', 'model', 'y_true', 'y_pred', 'fold']


,cell_id,mission_id,model,y_true,y_pred,fold
0,VAH01,1,LightGBM,99.652231,99.585068,1
1,VAH01,2,LightGBM,99.594270,99.538922,1



MODEL: CatBoost
Jumlah baris: 18513
Kolom: ['cell_id', 'mission_id', 'model', 'y_true', 'y_pred', 'cycle_instance_id', 'raw_cycle_number']


,cell_id,mission_id,model,y_true,y_pred,cycle_instance_id,raw_cycle_number
0,VAH01,1,CatBoost,99.652231,99.331747,1,0
1,VAH01,2,CatBoost,99.594270,99.237390,2,1


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path("/content/drive/MyDrive")

base = root / "eVTOL_table5_work/final_fast_loco_fixed"

files = {
    "Random Forest": "random_forest_predictions.csv",
    "XGBoost": "xgboost_predictions.csv",
    "LightGBM": "lightgbm_predictions.csv",
    "CatBoost": "catboost_loco_predictions.csv"
}

rows = []

for model, filename in files.items():
    df = pd.read_csv(base / filename)

    required = {"cell_id", "mission_id", "y_true", "y_pred"}
    assert required.issubset(df.columns), (
        f"{model}: kolom tidak lengkap"
    )

    assert df[["y_true", "y_pred"]].notna().all().all()

    duplicates = df.duplicated(
        ["cell_id", "mission_id"]
    ).sum()

    error = df["y_pred"] - df["y_true"]
    abs_error = error.abs()

    cell_mae = (
        df.assign(abs_error=abs_error)
        .groupby("cell_id")["abs_error"]
        .mean()
    )

    rows.append({
        "model": model,
        "samples": len(df),
        "cells": df["cell_id"].nunique(),
        "duplicates": int(duplicates),
        "pooled_MAE": abs_error.mean(),
        "pooled_RMSE": np.sqrt(np.mean(error**2)),
        "macro_MAE": cell_mae.mean(),
        "worst_cell_MAE": cell_mae.max()
    })

recomputed = pd.DataFrame(rows)

paper = pd.read_csv(
    root /
    "eVTOL_manuscript_results/"
    "soh_six_model/"
    "table1_six_model_soh_metrics.csv"
)

compare = recomputed.merge(
    paper[
        [
            "model",
            "pooled_MAE",
            "pooled_RMSE",
            "macro_MAE",
            "worst_cell_MAE"
        ]
    ],
    on="model",
    suffixes=("_recomputed", "_paper"),
    validate="one_to_one"
)

metrics = [
    "pooled_MAE",
    "pooled_RMSE",
    "macro_MAE",
    "worst_cell_MAE"
]

for metric in metrics:
    compare[f"diff_{metric}"] = (
        compare[f"{metric}_recomputed"]
        - compare[f"{metric}_paper"]
    ).abs()

compare["max_abs_diff"] = compare[
    [f"diff_{m}" for m in metrics]
].max(axis=1)

compare["numeric_match"] = (
    compare["max_abs_diff"] < 1e-5
)

print("HASIL REKONSILIASI MODEL POHON:")
display(
    compare[
        [
            "model",
            "samples",
            "cells",
            "duplicates",
            "pooled_MAE_recomputed",
            "pooled_MAE_paper",
            "max_abs_diff",
            "numeric_match"
        ]
    ].round(6)
)

print("\nSTATUS KESELURUHAN:")
print(
    "Semua cocok:",
    compare["numeric_match"].all()
)

HASIL REKONSILIASI MODEL POHON:


,model,samples,cells,duplicates,pooled_MAE_recomputed,pooled_MAE_paper,max_abs_diff,numeric_match
0,Random Forest,18513,19,0,0.599901,0.599901,0.000000,True
1,XGBoost,18513,19,0,0.571072,0.571072,0.000000,True
2,LightGBM,18513,19,0,0.525101,0.525101,0.000000,True
3,CatBoost,18513,19,0,0.899283,0.804183,0.229313,False



STATUS KESELURUHAN:
Semua cocok: False


In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path("/content/drive/MyDrive/eVTOL_table5_work")

candidates = {
    "final_fast":
        root / "final_fast/catboost_predictions.csv",
    "final_fast_loco_fixed":
        root / "final_fast_loco_fixed/catboost_loco_predictions.csv"
}

for label, path in candidates.items():
    print("\n" + "=" * 60)
    print("VERSI:", label)
    print("FILE:", path)
    print("ADA:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)

    print("Kolom:", df.columns.tolist())
    print("Baris:", len(df))

    if {"cell_id", "y_true", "y_pred"}.issubset(df.columns):
        error = df["y_pred"] - df["y_true"]
        cell_mae = (
            df.assign(abs_error=error.abs())
            .groupby("cell_id")["abs_error"]
            .mean()
        )

        print("Jumlah sel:", df["cell_id"].nunique())
        print("Pooled MAE:", round(error.abs().mean(), 6))
        print("Pooled RMSE:", round(np.sqrt(np.mean(error**2)), 6))
        print("Macro MAE:", round(cell_mae.mean(), 6))
        print("Worst-cell MAE:", round(cell_mae.max(), 6))

    display(df.head(2))


VERSI: final_fast
FILE: /content/drive/MyDrive/eVTOL_table5_work/final_fast/catboost_predictions.csv
ADA: True
Kolom: ['cell_id', 'mission_id', 'model', 'y_true', 'y_pred']
Baris: 18513
Jumlah sel: 19
Pooled MAE: 0.804183
Pooled RMSE: 1.324482
Macro MAE: 0.712724
Worst-cell MAE: 2.952953


,cell_id,mission_id,model,y_true,y_pred
0,VAH01,1,CatBoost,99.652231,99.301594
1,VAH01,2,CatBoost,99.594270,99.275536



VERSI: final_fast_loco_fixed
FILE: /content/drive/MyDrive/eVTOL_table5_work/final_fast_loco_fixed/catboost_loco_predictions.csv
ADA: True
Kolom: ['cell_id', 'mission_id', 'model', 'y_true', 'y_pred', 'cycle_instance_id', 'raw_cycle_number']
Baris: 18513
Jumlah sel: 19
Pooled MAE: 0.899283
Pooled RMSE: 1.553795
Macro MAE: 0.733759
Worst-cell MAE: 2.916567


,cell_id,mission_id,model,y_true,y_pred,cycle_instance_id,raw_cycle_number
0,VAH01,1,CatBoost,99.652231,99.331747,1,0
1,VAH01,2,CatBoost,99.594270,99.237390,2,1


In [23]:
from pathlib import Path
import json

root = Path(
    "/content/drive/MyDrive/eVTOL_table5_work"
)

files = [
    root / "final_fast_loco_fixed/experiment_manifest.json",
    root / "final_fast_loco_fixed/catboost_feature_manifest.json"
]

for path in files:
    print("\n" + "=" * 70)
    print("FILE:", path.relative_to(root))
    print("ADA:", path.exists())

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(json.dumps(data, indent=2, ensure_ascii=False)[:14000])

print("\n" + "=" * 70)
print("FILE AUDIT FITUR CATBOOST:")

for folder in [
    root / "final_fast",
    root / "final_fast_loco_fixed"
]:
    print("\nFOLDER:", folder.name)

    for path in sorted(folder.glob("*feature*")):
        print(" -", path.name)


FILE: final_fast_loco_fixed/experiment_manifest.json
ADA: True
{
  "source": "/content/drive/MyDrive/eVTOL_Table3_source_recovery/health_mission_features_targets.csv",
  "target": "SOH_m_plus_5_pct",
  "validation": "Leave-One-Cell-Out",
  "n_samples": 18513,
  "n_cells": 19,
  "features": [
    "chg_total_dur_s",
    "chg_cc_dur_s",
    "chg_cv_dur_s",
    "chg_energy_Wh",
    "chg_cc_energy_Wh",
    "chg_cv_energy_Wh",
    "chg_cv_fraction",
    "cv_setpoint_V",
    "chg_mean_I_A",
    "chg_mean_C_rate",
    "chg_max_C_rate",
    "chg_cv_mean_I_A_proxy",
    "dis_total_dur_s",
    "dis_flight_Wh_abs",
    "dis_mean_C_rate",
    "dis_mean_C_rate_med",
    "dis_max_C_rate",
    "dis_max_C_rate_p90",
    "dis_vi_R_ohm_med",
    "dis_vi_R_ohm_p90",
    "temp_mean_C",
    "temp_mean_C_med",
    "temp_max_C_max",
    "temp_th_gain_med",
    "temp_th_gain_max",
    "temp_slope_med",
    "gap_count",
    "q0_ah_audit",
    "chg_cum_Wh",
    "dis_cum_Wh"
  ],
  "excluded_columns": [
    "RUL

In [24]:
from pathlib import Path
import pandas as pd
import json

base = Path(
    "/content/drive/MyDrive/"
    "eVTOL_table5_work/final_fast_loco_fixed"
)

paths = [
    base / "catboost_loco_fold_metrics.csv",
    base / "catboost_loco_summary.csv",
    base / "soh_feature_audit_candidates.json"
]

for path in paths:
    print("\n" + "=" * 65)
    print("FILE:", path.name)
    print("ADA:", path.exists())

    if not path.exists():
        continue

    if path.suffix == ".csv":
        df = pd.read_csv(path)
        print("Kolom:", df.columns.tolist())
        print("Jumlah baris:", len(df))
        print(df.head(5).to_string(index=False))

    elif path.suffix == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False
            )[:12000]
        )


FILE: catboost_loco_fold_metrics.csv
ADA: True
Kolom: ['model', 'fold', 'cell_id', 'n_samples', 'MAE', 'RMSE', 'train_cells']
Jumlah baris: 19
   model  fold cell_id  n_samples      MAE     RMSE  train_cells
CatBoost     1   VAH01        824 0.881742 1.061675           18
CatBoost     2   VAH02        606 0.405006 0.457464           18
CatBoost     3   VAH05       1475 0.657868 0.811536           18
CatBoost     4   VAH10       1395 0.514274 0.875033           18
CatBoost     5   VAH11       2198 2.373214 3.221587           18

FILE: catboost_loco_summary.csv
ADA: True
Kolom: ['model', 'cells', 'samples', 'pooled_MAE', 'pooled_RMSE', 'macro_MAE', 'macro_RMSE', 'P90_cell_MAE', 'worst_cell_MAE', 'worst_cell_id', 'mean_cell_MAE_std']
Jumlah baris: 1
   model  cells  samples  pooled_MAE  pooled_RMSE  macro_MAE  macro_RMSE  P90_cell_MAE  worst_cell_MAE worst_cell_id  mean_cell_MAE_std
CatBoost     19    18513    0.899283     1.553795   0.733759    0.929688      1.568397        2.916567    

In [26]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

notebooks = sorted(
    p for p in root.rglob("*.ipynb")
    if p.is_file()
)

print("TOTAL NOTEBOOK:", len(notebooks))

for p in notebooks:
    print(p.relative_to(root))

TOTAL NOTEBOOK: 12
Colab Notebooks/Copy of eVTOL Li-Ion Battery.ipynb
Colab Notebooks/TUD_RL_UAM_Waltz2024.ipynb
Colab Notebooks/Untitled0.ipynb
Colab Notebooks/Untitled2.ipynb
Colab Notebooks/eVTOL Li-Ion Battery.ipynb
Colab Notebooks/eVTOL_Liu2025_(PENGERJAAN).ipynb
Colab Notebooks/eVTOL_SOH_Final_Experiments.ipynb
Colab Notebooks/evtol_replication_colab.ipynb
Colab Notebooks/evtol_revision_audit.ipynb
Colab Notebooks/replikasi_table4_actrajnet.ipynb
TUD_RL_UAM_Waltz2024.ipynb
eVTOL_Liu2025_(PENGERJAAN).ipynb


In [27]:
from pathlib import Path
import json

root = Path("/content/drive/MyDrive/Colab Notebooks")

names = [
    "eVTOL_SOH_Final_Experiments.ipynb",
    "evtol_replication_colab.ipynb",
    "evtol_revision_audit.ipynb"
]

keywords = [
    "catboost",
    "catboost_predictions.csv",
    "catboost_loco_predictions.csv",
    "catboost_feature_manifest",
    "final_fast_loco_fixed",
    "final_fast"
]

for name in names:
    path = root / name

    print("\n" + "=" * 75)
    print("NOTEBOOK:", name)
    print("ADA:", path.exists())

    if not path.exists():
        continue

    with open(path, encoding="utf-8") as f:
        nb = json.load(f)

    matches = []

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        found = [
            word for word in keywords
            if word.lower() in source.lower()
        ]

        if found:
            matches.append((i, found, source))

    print("SEL KODE RELEVAN:", len(matches))

    for i, found, source in matches[:12]:
        print(f"\n--- SEL {i} | KATA KUNCI: {found} ---")
        print(source[:1800])
        if len(source) > 1800:
            print("... [potongan kode dipersingkat]")


NOTEBOOK: eVTOL_SOH_Final_Experiments.ipynb
ADA: True
SEL KODE RELEVAN: 20

--- SEL 1 | KATA KUNCI: ['catboost'] ---
import os
import sys
import platform

print("Python version :", sys.version)
print("Platform       :", platform.platform())
print("Working dir    :", os.getcwd())

try:
    import sklearn
    print("scikit-learn   :", sklearn.__version__)
except ImportError:
    print("scikit-learn belum terpasang")

try:
    import xgboost
    print("XGBoost        :", xgboost.__version__)
except ImportError:
    print("XGBoost belum terpasang")

try:
    import lightgbm
    print("LightGBM       :", lightgbm.__version__)
except ImportError:
    print("LightGBM belum terpasang")

try:
    import catboost
    print("CatBoost       :", catboost.__version__)
except ImportError:
    print("CatBoost belum terpasang")

--- SEL 82 | KATA KUNCI: ['catboost'] ---
from pathlib import Path
import zipfile
import re

DRIVE = Path("/content/drive/MyDrive")

REPO = DRIVE / "Mission-Phase-Feature-Lear

In [31]:
%pip install -q catboost

In [32]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

root = Path("/content/drive/MyDrive")

source = (
    root / "eVTOL_Table3_source_recovery"
    / "health_mission_features_targets.csv"
)

manifest_path = (
    root / "eVTOL_table5_work"
    / "final_fast_loco_fixed"
    / "experiment_manifest.json"
)

out = root / "eVTOL_table5_work" / "corrected_train_only"
out.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(source)
manifest = json.loads(manifest_path.read_text())

features = manifest["features"]
target = "SOH_m_plus_5_pct"

data = df.dropna(subset=[target, "cell_id"]).copy()
X = data[features].replace([np.inf, -np.inf], np.nan)
y = data[target].astype(float)
cells = data["cell_id"].astype(str)

assert len(features) == 30
assert cells.nunique() == 19
assert len(data) == 18513

predictions = []
fold_metrics = []

for fold, test_cell in enumerate(sorted(cells.unique()), 1):

    train_idx = cells != test_cell
    test_idx = cells == test_cell

    # Imputer hanya belajar dari sel pelatihan
    imputer = SimpleImputer(strategy="median")

    X_train = imputer.fit_transform(X.loc[train_idx])
    X_test = imputer.transform(X.loc[test_idx])

    model = CatBoostRegressor(
        iterations=700,
        depth=7,
        learning_rate=0.04,
        loss_function="RMSE",
        verbose=False,
        random_seed=42,
        thread_count=-1
    )

    model.fit(X_train, y.loc[train_idx])

    yp = model.predict(X_test)
    yt = y.loc[test_idx].to_numpy()

    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))

    part = data.loc[
        test_idx, ["cell_id", "mission_id"]
    ].copy()

    part["model"] = "CatBoost"
    part["y_true"] = yt
    part["y_pred"] = yp

    predictions.append(part)

    fold_metrics.append({
        "model": "CatBoost",
        "fold": fold,
        "cell_id": test_cell,
        "n_test": len(yt),
        "MAE": mae,
        "RMSE": rmse,
        "train_cells": cells.loc[train_idx].nunique()
    })

    print(
        f"Fold {fold:02d}/19 | {test_cell} "
        f"| MAE={mae:.6f}"
    )

pp = pd.concat(predictions, ignore_index=True)
fm = pd.DataFrame(fold_metrics)

cell_mae = fm["MAE"]

summary = {
    "model": "CatBoost",
    "cells": int(pp.cell_id.nunique()),
    "samples": len(pp),
    "pooled_MAE": mean_absolute_error(pp.y_true, pp.y_pred),
    "pooled_RMSE": np.sqrt(
        mean_squared_error(pp.y_true, pp.y_pred)
    ),
    "macro_MAE": cell_mae.mean(),
    "P90_cell_MAE": cell_mae.quantile(0.90),
    "worst_cell_MAE": cell_mae.max()
}

pp.to_csv(out / "catboost_predictions.csv", index=False)
fm.to_csv(out / "catboost_fold_metrics.csv", index=False)

pd.DataFrame([summary]).to_csv(
    out / "catboost_summary.csv", index=False
)

print("\nHASIL CATBOOST DENGAN IMPUTASI TRAIN-ONLY")
print(pd.Series(summary).to_string())

print("\nHASIL LAMA MANUSKRIP")
print("Pooled MAE : 0.804183")
print("Pooled RMSE: 1.324482")
print("Macro MAE  : 0.712724")
print("Worst-cell : 2.952953")

print("\nOutput:", out)

Fold 01/19 | VAH01 | MAE=1.030398
Fold 02/19 | VAH02 | MAE=0.273351
Fold 03/19 | VAH05 | MAE=0.682432
Fold 04/19 | VAH10 | MAE=0.489351
Fold 05/19 | VAH11 | MAE=1.830889
Fold 06/19 | VAH12 | MAE=0.940209
Fold 07/19 | VAH13 | MAE=0.522814
Fold 08/19 | VAH15 | MAE=0.167016
Fold 09/19 | VAH16 | MAE=0.381336
Fold 10/19 | VAH17 | MAE=0.596424
Fold 11/19 | VAH20 | MAE=0.613047
Fold 12/19 | VAH22 | MAE=0.125157
Fold 13/19 | VAH23 | MAE=2.952953
Fold 14/19 | VAH24 | MAE=0.419302
Fold 15/19 | VAH25 | MAE=0.813301
Fold 16/19 | VAH26 | MAE=0.229131
Fold 17/19 | VAH27 | MAE=0.344677
Fold 18/19 | VAH28 | MAE=0.136890
Fold 19/19 | VAH30 | MAE=0.993081

HASIL CATBOOST DENGAN IMPUTASI TRAIN-ONLY
model             CatBoost
cells                   19
samples              18513
pooled_MAE        0.804183
pooled_RMSE       1.324482
macro_MAE         0.712724
P90_cell_MAE      1.190496
worst_cell_MAE    2.952953

HASIL LAMA MANUSKRIP
Pooled MAE : 0.804183
Pooled RMSE: 1.324482
Macro MAE  : 0.712724
Worst-c

In [33]:
from pathlib import Path
import json

root = Path("/content/drive/MyDrive/Colab Notebooks")

terms = [
    "q0_ah_audit",
    "chg_cum_Wh",
    "dis_cum_Wh"
]

notebooks = sorted(root.glob("*.ipynb"))

for path in notebooks:
    nb = json.loads(path.read_text(encoding="utf-8"))

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        matches = [
            j for j, line in enumerate(lines)
            if any(term in line for term in terms)
        ]

        if not matches:
            continue

        print("\n" + "=" * 75)
        print("NOTEBOOK:", path.name)
        print("SEL:", i)

        shown = set()

        for j in matches[:8]:
            for k in range(max(0, j - 5), min(len(lines), j + 6)):
                if k not in shown:
                    print(f"{k+1:03d} | {lines[k]}")
                    shown.add(k)
            print("-" * 40)


NOTEBOOK: Copy of eVTOL Li-Ion Battery.ipynb
SEL: 30
031 |     "dis_max_C_rate",
032 |     "dis_vi_R_ohm_med",
033 |     "temp_mean_C",
034 |     "temp_max_C_max",
035 |     "gap_count",
036 |     "chg_cum_Wh",
037 |     "dis_cum_Wh",
038 | ]
039 | 
040 | mission = pd.read_csv(REPO / "health_mission_features_targets.csv", nrows=1)
041 | mission_columns = set(mission.columns)
----------------------------------------
042 | 
----------------------------------------

NOTEBOOK: eVTOL Li-Ion Battery.ipynb
SEL: 30
031 |     "dis_max_C_rate",
032 |     "dis_vi_R_ohm_med",
033 |     "temp_mean_C",
034 |     "temp_max_C_max",
035 |     "gap_count",
036 |     "chg_cum_Wh",
037 |     "dis_cum_Wh",
038 | ]
039 | 
040 | mission = pd.read_csv(REPO / "health_mission_features_targets.csv", nrows=1)
041 | mission_columns = set(mission.columns)
----------------------------------------
042 | 
----------------------------------------

NOTEBOOK: eVTOL_SOH_Final_Experiments.ipynb
SEL: 31
031 |     "dis_max_

In [36]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive")

DATASET = (
    ROOT
    / "eVTOL_Table3_source_recovery"
    / "health_mission_features_targets.csv"
)

COMMIT = "253d59e5bf030d10480c50654db5bb620dc2fe44"

URL = (
    "https://raw.githubusercontent.com/"
    "MusaWiston/"
    "Mission-Phase-Feature-Learning-for-eVTOL-Battery-Prognostics-with-ML/"
    f"{COMMIT}/capacity_tests.csv"
)

print("=" * 80)
print("TAHAP 2.5I — VERIFIKASI KRONOLOGI RPT PERTAMA")
print("=" * 80)

# Baca data RPT dari commit benchmark yang tercatat
rpt = pd.read_csv(URL)

# Baca nilai kapasitas referensi dari dataset eksperimen
df = pd.read_csv(
    DATASET,
    usecols=["cell_id", "q0_ah_audit"]
)

print("\nJumlah baris RPT:", len(rpt))
print("Jumlah sel RPT :", rpt["cell_id"].nunique())

required = {
    "cell_id",
    "mission_position",
    "capacity_ah",
    "rpt_order",
}

assert required.issubset(rpt.columns), (
    f"Kolom RPT tidak lengkap: {required - set(rpt.columns)}"
)

# Pilih RPT pertama untuk setiap sel
first_rpt = (
    rpt.sort_values(
        ["cell_id", "rpt_order", "mission_position"]
    )
    .groupby("cell_id", as_index=False)
    .first()
)

first_rpt = first_rpt[
    ["cell_id", "mission_position", "capacity_ah"]
].rename(
    columns={
        "mission_position": "first_RPT_position",
        "capacity_ah": "q0_from_RPT",
    }
)

# Ambil nilai Q0 yang digunakan dalam dataset
q0_dataset = (
    df.groupby("cell_id", as_index=False)
    .agg(q0_dataset=("q0_ah_audit", "first"))
)

# Bandingkan Q0 dataset dengan RPT pertama
audit = q0_dataset.merge(
    first_rpt,
    on="cell_id",
    how="left",
    validate="one_to_one"
)

audit["q0_match"] = np.isclose(
    audit["q0_dataset"],
    audit["q0_from_RPT"],
    rtol=1e-6,
    atol=1e-6,
    equal_nan=False
)

audit["RPT_before_first_mission"] = (
    audit["first_RPT_position"] == 0
)

print("\nHASIL PER SEL:\n")
print(audit.to_string(index=False))

print("\n" + "=" * 80)
print("RINGKASAN")
print("=" * 80)

print("Sel benchmark:", len(audit))

print(
    "RPT pertama pada posisi 0:",
    int(audit["RPT_before_first_mission"].sum())
)

print(
    "RPT pertama setelah misi dimulai:",
    int((audit["first_RPT_position"] > 0).sum())
)

print(
    "Q0 cocok dengan dataset:",
    int(audit["q0_match"].sum())
)

print(
    "Sel tanpa data RPT:",
    int(audit["first_RPT_position"].isna().sum())
)

print("\nAUDIT SELESAI — TANPA TRAINING")

TAHAP 2.5I — VERIFIKASI KRONOLOGI RPT PERTAMA

Jumlah baris RPT: 441
Jumlah sel RPT : 22

HASIL PER SEL:

cell_id  q0_dataset  first_RPT_position  q0_from_RPT  q0_match  RPT_before_first_mission
  VAH01    3.089338                   0     3.089338      True                      True
  VAH02    3.104820                   0     3.104820      True                      True
  VAH05    3.033223                   0     3.033223      True                      True
  VAH10    3.085544                   0     3.085544      True                      True
  VAH11    3.049507                   0     3.049507      True                      True
  VAH12    3.060796                   0     3.060796      True                      True
  VAH13    3.045314                   0     3.045314      True                      True
  VAH15    3.124816                   0     3.124816      True                      True
  VAH16    3.138379                   0     3.138379      True                      True
  VA

In [37]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path("/content/drive/MyDrive/eVTOL_table5_work")

models = {
    "Random Forest": "random_forest_predictions.csv",
    "XGBoost": "xgboost_predictions.csv",
    "LightGBM": "lightgbm_predictions.csv",
}

folder = root / "final_fast_loco_fixed"

for model, filename in models.items():
    path = folder / filename

    print("\n" + "=" * 65)
    print(model)
    print("File:", path)

    if not path.exists():
        print("FILE TIDAK DITEMUKAN")
        continue

    df = pd.read_csv(path)

    required = {"cell_id", "y_true", "y_pred"}
    if not required.issubset(df.columns):
        print("KOLOM TIDAK SESUAI:", list(df.columns))
        continue

    error = df["y_true"] - df["y_pred"]

    per_cell = (
        df.assign(abs_error=error.abs())
        .groupby("cell_id")["abs_error"]
        .mean()
    )

    print("Prediksi:", len(df))
    print("Sel uji:", df["cell_id"].nunique())
    print("Pooled MAE:", round(error.abs().mean(), 6))
    print("Pooled RMSE:", round(np.sqrt(np.mean(error**2)), 6))
    print("Macro MAE:", round(per_cell.mean(), 6))
    print("Worst-cell MAE:", round(per_cell.max(), 6))



Random Forest
File: /content/drive/MyDrive/eVTOL_table5_work/final_fast_loco_fixed/random_forest_predictions.csv
Prediksi: 18513
Sel uji: 19
Pooled MAE: 0.599901
Pooled RMSE: 0.85242
Macro MAE: 0.591775
Worst-cell MAE: 1.82365

XGBoost
File: /content/drive/MyDrive/eVTOL_table5_work/final_fast_loco_fixed/xgboost_predictions.csv
Prediksi: 18513
Sel uji: 19
Pooled MAE: 0.571072
Pooled RMSE: 0.82429
Macro MAE: 0.582537
Worst-cell MAE: 2.094553

LightGBM
File: /content/drive/MyDrive/eVTOL_table5_work/final_fast_loco_fixed/lightgbm_predictions.csv
Prediksi: 18513
Sel uji: 19
Pooled MAE: 0.525101
Pooled RMSE: 0.759373
Macro MAE: 0.544597
Worst-cell MAE: 1.924715


In [39]:
from pathlib import Path
import json

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "eVTOL_SOH_Final_Experiments.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

for i in [127, 129]:
    cell = nb["cells"][i]
    source = "".join(cell.get("source", []))

    print("\n" + "=" * 80)
    print(f"SEL {i}")
    print("=" * 80)

    for number, line in enumerate(source.splitlines(), 1):
        print(f"{number:03d} | {line}")


SEL 127
001 | # ============================================================
002 | # CELL 105 — TABLE 5: LEAKAGE-AWARE LOCO SOH BENCHMARK
003 | # Models: Random Forest, XGBoost, LightGBM
004 | # Target: SOH after 5 missions
005 | # Validation: Leave-One-Cell-Out (LOCO)
006 | # ============================================================
007 | 
008 | from pathlib import Path
009 | import time
010 | import warnings
011 | import json
012 | 
013 | import numpy as np
014 | import pandas as pd
015 | 
016 | from sklearn.base import clone
017 | from sklearn.impute import SimpleImputer
018 | from sklearn.ensemble import RandomForestRegressor
019 | from sklearn.metrics import mean_absolute_error, mean_squared_error
020 | 
021 | warnings.filterwarnings("ignore")
022 | 
023 | # ------------------------------------------------------------
024 | # 1. PATHS AND CONFIGURATION
025 | # ------------------------------------------------------------
026 | 
027 | SRC = Path(
028 |     "/content/drive/MyDriv

In [40]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path("/content/drive/MyDrive/eVTOL_table5_work")

old = pd.read_csv(
    root / "final_fast" / "catboost_predictions.csv"
)

new = pd.read_csv(
    root / "corrected_train_only" / "catboost_predictions.csv"
)

keys = ["cell_id", "mission_id"]

assert not old.duplicated(keys).any(), "Duplikat pada prediksi lama"
assert not new.duplicated(keys).any(), "Duplikat pada prediksi koreksi"

merged = old.merge(
    new,
    on=keys,
    suffixes=("_old", "_new"),
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("Baris lama:", len(old))
print("Baris koreksi:", len(new))
print("Status pencocokan:")
print(merged["_merge"].value_counts().to_string())

both = merged[merged["_merge"] == "both"].copy()

true_diff = np.abs(
    both["y_true_old"] - both["y_true_new"]
)

pred_diff = np.abs(
    both["y_pred_old"] - both["y_pred_new"]
)

print("\nPerbandingan per prediksi:")
print("Maksimum beda y_true :", true_diff.max())
print("Maksimum beda y_pred :", pred_diff.max())
print("Rata-rata beda y_pred:", pred_diff.mean())
print("Prediksi berbeda > 1e-8:", int((pred_diff > 1e-8).sum()))
print("Prediksi berbeda > 1e-6:", int((pred_diff > 1e-6).sum()))

print("\nIdentik numerik (toleransi 1e-8):")
print(
    len(both) == len(old) == len(new)
    and np.allclose(true_diff, 0, atol=1e-8, rtol=0)
    and np.allclose(pred_diff, 0, atol=1e-8, rtol=0)
)

Baris lama: 18513
Baris koreksi: 18513
Status pencocokan:
_merge
both          18513
left_only         0
right_only        0

Perbandingan per prediksi:
Maksimum beda y_true : 0.0
Maksimum beda y_pred : 0.0
Rata-rata beda y_pred: 0.0
Prediksi berbeda > 1e-8: 0
Prediksi berbeda > 1e-6: 0

Identik numerik (toleransi 1e-8):
True


In [41]:
from pathlib import Path
import json
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/"
    "eVTOL_benchmark_reproduction_v2/table3_full_run/soh"
)

for model in ["mission_phase_transformer", "attention_lstm_moe"]:
    print("\n" + "=" * 80)
    print("MODEL:", model)
    print("=" * 80)

    folder = BASE / model

    for filename in ["fold_record.json", "scaler.json", "training_log.csv"]:
        matches = sorted(folder.rglob(filename))

        print(f"\n{filename}: {len(matches)} file ditemukan")

        if not matches:
            continue

        path = matches[0]
        print("CONTOH:", path.relative_to(BASE))

        try:
            if path.suffix == ".json":
                obj = json.loads(path.read_text(encoding="utf-8"))
                print(json.dumps(obj, indent=2, ensure_ascii=False)[:9000])
            else:
                df = pd.read_csv(path)
                print("Kolom:", df.columns.tolist())
                print(df.head(5).to_string(index=False))
                print("...")
                print(df.tail(5).to_string(index=False))
        except Exception as e:
            print("GAGAL BACA:", repr(e))


MODEL: mission_phase_transformer

fold_record.json: 19 file ditemukan
CONTOH: mission_phase_transformer/VAH01/fold_record.json
{
  "task": "soh",
  "model": "mission_phase_transformer",
  "fold": 1,
  "test_cell": "VAH01",
  "train_cells": "VAH05;VAH10;VAH11;VAH13;VAH15;VAH16;VAH17;VAH20;VAH22;VAH23;VAH24;VAH28",
  "validation_cells": "VAH02;VAH12;VAH25;VAH26;VAH27;VAH30",
  "train_n": 11744,
  "validation_n": 5945,
  "test_n": 824,
  "test_mae": 5.456125736236572,
  "test_rmse": 7.208609104156494,
  "inference_ms_per_sample": 0.2721281055824186,
  "best_epoch": 4,
  "best_val_mae": 1.3721601963043213,
  "epochs_run": 12,
  "training_seconds": 136.41291351200016,
  "parameter_count": 160417,
  "target_mean": 86.6927261352539,
  "target_std": 6.234044075012207
}

scaler.json: 19 file ditemukan
CONTOH: mission_phase_transformer/VAH01/scaler.json
{
  "median": [
    4406.9306640625,
    6.633365631103516,
    1.357464075088501,
    0.44525909423828125,
    1253.1981201171875,
    4.26856

In [42]:
from pathlib import Path
import json

root = Path("/content/drive/MyDrive/Colab Notebooks")
keywords = [
    "target_mean",
    "best_val_mae",
    "scaler.json",
    "nanmedian",
    "nanmean",
    "early_stopping",
]

for path in sorted(root.glob("*.ipynb")):
    nb = json.loads(path.read_text(encoding="utf-8"))

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        lines = "".join(cell.get("source", [])).splitlines()
        hits = [
            j for j, line in enumerate(lines)
            if any(k in line.lower() for k in keywords)
        ]

        if not hits:
            continue

        print("\n" + "=" * 75)
        print("NOTEBOOK:", path.name, "| SEL:", i)
        print("=" * 75)

        shown = set()
        for j in hits:
            for k in range(max(0, j - 6), min(len(lines), j + 9)):
                if k not in shown:
                    print(f"{k+1:03d} | {lines[k]}")
                    shown.add(k)


NOTEBOOK: Copy of eVTOL Li-Ion Battery.ipynb | SEL: 62
025 |         for fold in fold_dirs:
026 |             required = [
027 |                 fold / "best.pt",
028 |                 fold / "fold_record.json",
029 |                 fold / "predictions.csv",
030 |                 fold / "training_log.csv",
031 |                 fold / "scaler.json",
032 |             ]
033 | 
034 |             if all(p.exists() and p.stat().st_size > 0 for p in required):
035 |                 complete += 1
036 | 
037 |             if (fold / "predictions.csv").exists():
038 |                 predictions += 1
039 |             if (fold / "training_log.csv").exists():

NOTEBOOK: Copy of eVTOL Li-Ion Battery.ipynb | SEL: 69
066 |         mae=("abs_error", "mean"),
067 |     )
068 |     print(summary.round(3).to_string())
069 | 
070 |     print("\nContoh fold metadata:")
071 |     print(
072 |         "Target mean training =", record["target_mean"],
073 |         "| Target std training =", record["targe